# 05 — VCOD bootstrap, validation, and backbone mapping gate
Run this notebook from a fresh A100-class Colab GPU runtime. It downloads and caches the official MoCA and CAMotion releases in Drive, builds leakage-safe manifests, validates sparse manual targets and sequence-level CAMotion attributes, records review receipts, and verifies the exact DINOv3/V-JEPA2.1 pathways.


In [ ]:
# Fresh-kernel bootstrap. Edit only these settings; cached Drive assets are reused.
ACCEPT_CAMOTION_ACADEMIC_LICENSE = True #@param {type:'boolean'}
PROJECT_REPO_URL = 'https://github.com/papanag/cod-ssl.git'
PROJECT_BRANCH = 'main'
DRIVE_ROOT = '/content/drive/MyDrive/cod-ssl'

from google.colab import drive
drive.mount('/content/drive')
from getpass import getpass
from pathlib import Path
from collections import deque
import json, os, subprocess, sys, torch
os.environ['TQDM_MININTERVAL'] = '5'  # Keep Colab outputs compact during long Drive scans.

def run_streaming(command, *, cwd, env=None):
    child_env = os.environ.copy() if env is None else env.copy()
    child_env['PYTHONUNBUFFERED'] = '1'
    child_env['TQDM_MININTERVAL'] = '5'
    process = subprocess.Popen(
        command, cwd=cwd, env=child_env, text=True, bufsize=1,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    )
    tail = deque(maxlen=80)
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end='')
        tail.append(line)
    if process.wait():
        raise RuntimeError(
            f'Bootstrap command failed with exit status {process.returncode}.\n'
            f'Last child-process output:\n{"".join(tail)}'
        )

project_dir = Path('/content/cod-ssl')
if (project_dir / '.git').is_dir():
    subprocess.run(['git', '-C', str(project_dir), 'fetch', 'origin', PROJECT_BRANCH], check=True)
    subprocess.run(['git', '-C', str(project_dir), 'checkout', PROJECT_BRANCH], check=True)
    subprocess.run(['git', '-C', str(project_dir), 'pull', '--ff-only', 'origin', PROJECT_BRANCH], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', PROJECT_BRANCH, PROJECT_REPO_URL, str(project_dir)], check=True)
# Dataset/backbone validation does not require the CUDA-only mamba-ssm package.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', f'{project_dir}[dev,notebooks]'], check=True)

bootstrap_env = os.environ.copy()
dino_weights = Path(DRIVE_ROOT) / 'checkpoints/dinov3_vitb16.pth'
if not dino_weights.is_file():
    print('DINOv3 requires approved Meta access on the first run only.')
    private_url = getpass('Private DINOv3 ViT-B/16 LVD-1689M URL: ').strip()
    if not private_url: raise ValueError('The approved DINOv3 URL is required.')
    bootstrap_env['COD_SSL_DINOV3_DOWNLOAD_URL'] = private_url
    del private_url
state_file = Path('/content/cod_ssl_bootstrap_state.json')
subprocess.run([sys.executable, str(project_dir / 'scripts/bootstrap_colab.py'),
                '--project-dir', str(project_dir), '--drive-root', DRIVE_ROOT,
                '--state-file', str(state_file)],
               cwd=project_dir, env=bootstrap_env, check=True)
bootstrap_env.pop('COD_SSL_DINOV3_DOWNLOAD_URL', None)
state = json.loads(state_file.read_text())
os.environ.update(state['environment'])
PROJECT_DIR = Path(state['project_dir'])
VCOD_ROOT = Path(state['drive_root']) / 'vcod'
MOCA_MANIFEST = VCOD_ROOT / 'data/processed/moca_mask_dense_v1/manifest/runtime_manifest.csv'
CAMOTION_MANIFEST = VCOD_ROOT / 'manifests/camotion.csv'
INSPECTION_ROOT = VCOD_ROOT / 'inspections'
APPROVAL_ROOT = VCOD_ROOT / 'approvals'
for path in (INSPECTION_ROOT, APPROVAL_ROOT): path.mkdir(parents=True, exist_ok=True)
os.chdir(PROJECT_DIR)

# Public VCOD data bootstrap. CAMotion is academic-research-only.
if not ACCEPT_CAMOTION_ACADEMIC_LICENSE:
    raise PermissionError('Review the official CAMotion usage notice, then enable the acknowledgement above.')
run_streaming([
    sys.executable, 'scripts/bootstrap_vcod_data.py',
    '--data-root', str(VCOD_ROOT / 'data'),
    '--manifest-dir', str(VCOD_ROOT / 'manifests'),
    '--staging-root', '/content/vcod_extract_staging',
    '--datasets', 'moca_mask_dense', 'camotion',
    '--accept-camotion-academic-license',
], cwd=PROJECT_DIR)
os.environ['MOCA_MASK_DENSE_MANIFEST'] = str(MOCA_MANIFEST)
os.environ['CAMOTION_MANIFEST'] = str(CAMOTION_MANIFEST)
print('Ready on', state['gpu'], 'with VCOD root', VCOD_ROOT)


In [ ]:
# Runtime and asset preflight. The locked primary protocol uses BF16 and a high-memory GPU.
if not torch.cuda.is_available(): raise RuntimeError('Select a GPU runtime.')
properties = torch.cuda.get_device_properties(0)
gpu_report = {
    'name': properties.name, 'memory_gib': round(properties.total_memory / 2**30, 2),
    'compute_capability': f'{properties.major}.{properties.minor}',
    'bf16_supported': torch.cuda.is_bf16_supported(),
}
print(json.dumps(gpu_report, indent=2))
if not gpu_report['bf16_supported']:
    raise RuntimeError('The locked BF16 protocol requires a BF16-capable GPU; request an A100-class runtime.')
for manifest in (MOCA_MANIFEST, CAMOTION_MANIFEST):
    if not manifest.is_file(): raise FileNotFoundError(manifest)


In [ ]:
# Repository correctness gate, cached for the exact commit and environment.
subprocess.run([sys.executable, 'scripts/run_cached_pytest.py',
                '--project-dir', str(PROJECT_DIR),
                '--receipt', str(INSPECTION_ROOT / 'repository_tests.json')],
               cwd=PROJECT_DIR, check=True)

In [ ]:
# Inspect both active primary datasets. Reports and visual QA persist in Drive.
inspection_commands = [
    ('moca_mask_dense', [sys.executable, 'scripts/inspect_dataset.py',
                   '--config', 'configs/datasets/moca_mask_dense.yaml', '--manifest', str(MOCA_MANIFEST),
                   '--output', str(INSPECTION_ROOT / 'moca_mask_dense'), '--workers', '16']),
    ('camotion', [sys.executable, 'scripts/inspect_dataset.py',
                  '--config', 'configs/datasets/camotion.yaml', '--manifest', str(CAMOTION_MANIFEST),
                  '--output', str(INSPECTION_ROOT / 'camotion'), '--workers', '16']),
]
for name, command in inspection_commands:
    print('Inspecting', name); subprocess.run(command, cwd=PROJECT_DIR, check=True)
if hasattr(os, 'sync'): os.sync()


In [ ]:
# Audit source identities and representative image overlap; reuse an exact cached result.
import pandas as pd
from cod_ssl.utils.run import file_sha256, git_commit
overlap_path = INSPECTION_ROOT / 'cross_dataset_overlap.json'
subprocess.run([sys.executable, 'scripts/audit_vcod_overlap.py',
                '--moca-manifest', str(MOCA_MANIFEST),
                '--camotion-manifest', str(CAMOTION_MANIFEST),
                '--camotion-attributes', str(VCOD_ROOT / 'manifests/camotion.attributes.json'),
                '--output', str(overlap_path)], cwd=PROJECT_DIR, check=True)
audit = json.loads(overlap_path.read_text())
moca, camotion = pd.read_csv(MOCA_MANIFEST), pd.read_csv(CAMOTION_MANIFEST)
if hasattr(os, 'sync'): os.sync()


In [ ]:
# Checkpoint-specific image/video pathway and dense-token mapping gate.
commands = [
    [sys.executable, 'scripts/inspect_backbone.py', '--model', 'dinov3_vitb16',
     '--pathway', 'image', '--output', str(INSPECTION_ROOT / 'backbone_dinov3.json')],
    [sys.executable, 'scripts/inspect_backbone.py', '--model', 'vjepa21_vitb16',
     '--pathway', 'image', 'video', '--clip-length', '64', '--target-index', '32',
     '--output', str(INSPECTION_ROOT / 'backbone_vjepa21.json')],
]
for command in commands:
    subprocess.run(command, cwd=PROJECT_DIR, check=True)
    gc = __import__('gc'); gc.collect(); torch.cuda.empty_cache()

In [ ]:
# Display automated reports and contact sheets for manual review.
from IPython.display import Markdown, display
from PIL import Image
for name, _ in inspection_commands:
    directory = INSPECTION_ROOT / name
    report_name = 'summary.md' if name == 'camotion' else 'report.md'
    display(Markdown((directory / report_name).read_text()))
    display(Image.open(directory / 'random_overlays.png'))


In [ ]:
# Inspect deterministic boundary padding and chronological source indices on real videos.
from cod_ssl.data.clip_sampler import ClipSampler, ClipSpec
from PIL import ImageDraw
def boundary_contact_sheet(frame, label):
    source_id = sorted(frame.source_video_id.astype(str).unique())[0]
    group = frame[frame.source_video_id.astype(str) == source_id].sort_values('frame_number').reset_index(drop=True)
    sampler, spec = ClipSampler(), ClipSpec(5, 1, 2)
    panels = []
    for target_position in (0, len(group) - 1):
        positions, valid = sampler.source_indices(group.frame_number.astype(int).tolist(), target_position, spec)
        for slot, (position, is_valid) in enumerate(zip(positions, valid.tolist())):
            row = group.iloc[position]
            with Image.open(row.image_path) as raw: image = raw.convert('RGB')
            image.thumbnail((180, 130))
            panel = Image.new('RGB', (190, 160), 'white'); panel.paste(image, ((190-image.width)//2, 0))
            ImageDraw.Draw(panel).text((4, 136), f'target={target_position} slot={slot} frame={row.frame_number} valid={is_valid}', fill='black')
            panels.append(panel)
    sheet = Image.new('RGB', (5 * 190, 2 * 160), 'white')
    for index, panel in enumerate(panels): sheet.paste(panel, ((index % 5) * 190, (index // 5) * 160))
    path = INSPECTION_ROOT / f'{label}_boundary_clips.png'; sheet.save(path)
    print(label, 'source_video_id=', source_id); display(sheet)
boundary_contact_sheet(moca[moca.split == 'train'], 'moca_mask_dense')
boundary_contact_sheet(camotion[camotion.split == 'train'], 'camotion')
if hasattr(os, 'sync'): os.sync()


In [ ]:
#@title Manual correctness sign-off (edit every field after inspecting the outputs)
REVIEWER = '' #@param {type:'string'}
DATASET_RELEASES = '' #@param {type:'string'}
OVERLAYS_ALIGNED = False #@param {type:'boolean'}
BOUNDARY_CLIPS_CORRECT = False #@param {type:'boolean'}
VJEPA_TUBELET_MAPPING_APPROVED = False #@param {type:'boolean'}
FEATURE_ORIENTATION_APPROVED = False #@param {type:'boolean'}
WARNINGS_ACKNOWLEDGED = False #@param {type:'boolean'}

checks = [OVERLAYS_ALIGNED, BOUNDARY_CLIPS_CORRECT, VJEPA_TUBELET_MAPPING_APPROVED,
          FEATURE_ORIENTATION_APPROVED, WARNINGS_ACKNOWLEDGED]
if not REVIEWER.strip() or not DATASET_RELEASES.strip() or not all(checks):
    raise PermissionError('Complete every manual sign-off field before authorizing training.')
from datetime import datetime, timezone
evidence_paths = {
    'moca_inspection_receipt': INSPECTION_ROOT / 'moca_mask_dense/inspection_receipt.json',
    'camotion_inspection_receipt': INSPECTION_ROOT / 'camotion/inspection_receipt.json',
    'cross_dataset_overlap': INSPECTION_ROOT / 'cross_dataset_overlap.json',
    'repository_tests': INSPECTION_ROOT / 'repository_tests.json',
    'dinov3_backbone': INSPECTION_ROOT / 'backbone_dinov3.json',
    'vjepa21_backbone': INSPECTION_ROOT / 'backbone_vjepa21.json',
    'moca_boundary_clips': INSPECTION_ROOT / 'moca_mask_dense_boundary_clips.png',
    'camotion_boundary_clips': INSPECTION_ROOT / 'camotion_boundary_clips.png',
}
missing_evidence = [str(path) for path in evidence_paths.values() if not path.is_file()]
if missing_evidence: raise FileNotFoundError(f'Missing validation evidence: {missing_evidence}')
approval = {
    'schema_version': 2, 'project_commit': git_commit(PROJECT_DIR),
    'reviewer': REVIEWER.strip(), 'dataset_releases': DATASET_RELEASES.strip(),
    'approved_at_utc': datetime.now(timezone.utc).isoformat(),
    'moca_manifest_sha256': file_sha256(MOCA_MANIFEST),
    'camotion_manifest_sha256': file_sha256(CAMOTION_MANIFEST),
    'camotion_attribute_manifest_sha256': file_sha256(VCOD_ROOT / 'manifests/camotion.attributes.json'),
    'configuration_sha256': {
        name: file_sha256(PROJECT_DIR / path) for name, path in {
            'moca_mask_dense': 'configs/datasets/moca_mask_dense.yaml',
            'camotion': 'configs/datasets/camotion.yaml',
        }.items()
    },
    'evidence_sha256': {name: file_sha256(path) for name, path in evidence_paths.items()},
    'overlays_aligned': OVERLAYS_ALIGNED, 'boundary_clips_correct': BOUNDARY_CLIPS_CORRECT,
    'vjepa_tubelet_mapping_approved': VJEPA_TUBELET_MAPPING_APPROVED,
    'feature_orientation_approved': FEATURE_ORIENTATION_APPROVED,
    'warnings_acknowledged': WARNINGS_ACKNOWLEDGED,
}
approval_path = APPROVAL_ROOT / 'vcod_validation_approval.json'
approval_temporary = approval_path.with_suffix('.json.tmp')
approval_temporary.write_text(json.dumps(approval, indent=2) + '\n')
approval_temporary.replace(approval_path)
if hasattr(os, 'sync'): os.sync()
print('Available-dataset training gate approved:', approval_path)
